<a href="https://colab.research.google.com/github/sruthi-analyst/sruthi-codeboosters-2026/blob/main/Day7/Day_7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Keyword search fails
embeddings as a solution
chromaDB vector
##Key word search Fails
1. Query = 'How do vehicles move?'
  - it exactly looks for the keyword - 'vehicle'
  - the data might not contain all the relevant answers associated with the same keyword, the information will be associated with different similar closely related keywords like - 'car', 'train', 'engined locomotive machines'
  - To avoid missing all synonymous and related words - avoid using 'keyword search'
##Semantic search
1. Query = 'How do vehicles move?'
  - embedding - vectorisation - a way to represent synonymous words (closely related words) closer together in a space of coordinates - semantic space
  - the numbers - semantic numbers
2. Search: vehicles - 0.8
  - cars - 0.81
  both are closely related which makes it find information involving text of the embedding 0.8 and 0.81
  ### Embeddings
  A long list of numbers format of a word
##Vector DB - Chroma DB
Embedding - converts strings into 'set of integers' - vectors
stores the vectors into a vector DB
e.g., Chroma DB
####What is ChromaDB?
  ChromaDB is an open-source vector database designed for storing, indexing, and querying embeddings (vectors) efficiently. In the context of large language models (LLMs) and semantic search, embeddings are numerical representations of text, images, audio, or other data, capturing their semantic meaning. ChromaDB allows you to store these high-dimensional vectors and perform fast similarity searches to find items that are semantically related.

####Is it a library?
  Yes, ChromaDB provides a Python client library (and clients for other languages like JavaScript) that allows you to interact with the database programmatically. This means you can integrate ChromaDB directly into your Python applications to:
    *   Create collections: Organize your vectors into logical groups.
    *   Add embeddings: Insert new vectors along with their associated metadata.
    *   Query the database: Perform similarity searches (e.g., find the 'n' most similar vectors to a given query vector).
    *   Manage data: Update or delete vectors and their metadata.

####How does it work?
  - Embeddings: You first generate embeddings for your data using an embedding model (e.g., from OpenAI, Hugging Face, or Google's models). These embeddings are dense vectors (lists of numbers).
  - Storage: You then use the ChromaDB client library to add these embeddings to a collection within your ChromaDB instance. ChromaDB stores these vectors and their associated original text or metadata.
  - Indexing: Internally, ChromaDB uses efficient data structures and algorithms (like HNSW - Hierarchical Navigable Small World) to index these vectors, enabling fast approximate nearest neighbor (ANN) searches.
  - Querying: When you have a new query (e.g., a user's question), you convert that query into an embedding using the same embedding model. You then send this query embedding to ChromaDB, which quickly finds and returns the most similar stored embeddings (and their original content/metadata).

####Key features of ChromaDB:
  - Vector storage and indexing: Optimized for high-dimensional vector data.
  - Similarity search: Supports various similarity metrics (e.g., cosine similarity).
  - Metadata filtering: Allows you to filter search results based on associated metadata.
  - Scalability: Can handle large datasets of embeddings.
  - Ease of use: Simple API for common operations.
  - Open-source: Transparent and community-driven development.

  In essence, ChromaDB acts as the backend for your semantic search, recommendation systems, or RAG (Retrieval Augmented Generation) applications, providing the infrastructure to make sense of your data through vector embeddings.

In [2]:
#installing 2 libraries: chromadb, and sentence-transformers
#chromadb: vector databased (like SQLite but for AI embedding)
#sentence-transformers: converts text to 384-dimensional number vectors

!pip install chromadb sentence-transformers -q
#-q means 'quiet mode' - reduces the amount of output printed
#You will see a progress bar. Wait until it says 'Successfully installed'

print("Dependencies Imported!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 69.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 109.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 81.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the 

In [3]:
import pandas as pd
import numpy as np

#new library: sentence_transformers
# SentenceTransformer: the clas we use to load the embeddings model
from sentence_transformers import SentenceTransformer

# New library: chromaDB
# chromaDB: the vector database package
import chromadb

print("Imported numpy & pandas")
print(f"ChromaDB version: {chromadb.__version__}")

Imported numpy & pandas
ChromaDB version: 1.5.9


In [4]:
#DEMO: Keyword search vs Semantic Search

#Imaging we have a  small collection of documents about data
documents = [
    "ETL is used to clean and transform data",
    "A vehicle is a mode of transport",
    "Cars and truck are popular automobiles",
    "SQL is used to query databses",
    "Machine learning trains models on data",
    ]
query_keyword = "vehicle" #the word we are searching for

print("~="*30)
print(f"\t\tKEYWORD SEARCH for '{query_keyword}'")
print("~="*30)
print()

for i,doc in enumerate(documents):
  if query_keyword.lower() in doc.lower():
    print(f"    FOUND\t[doc_{i}]: {doc}")
  else:
    print(f"    MISSED\t[doc_{i}]: {doc}")

print()
print("PROBLEM: doc_2 talks about 'Cars and Trucks' - which ARE vehicle!")
print("But keyword search MISSED it because it searched for the xeact word")

~=~=~=~=~=~=~=~=~=~=~=~=~=~=~=~=~=~=~=~=~=~=~=~=~=~=~=~=~=~=
		KEYWORD SEARCH for 'vehicle'
~=~=~=~=~=~=~=~=~=~=~=~=~=~=~=~=~=~=~=~=~=~=~=~=~=~=~=~=~=~=

    MISSED	[doc_0]: ETL is used to clean and transform data
    FOUND	[doc_1]: A vehicle is a mode of transport
    MISSED	[doc_2]: Cars and truck are popular automobiles
    MISSED	[doc_3]: SQL is used to query databses
    MISSED	[doc_4]: Machine learning trains models on data

PROBLEM: doc_2 talks about 'Cars and Trucks' - which ARE vehicle!
But keyword search MISSED it because it searched for the xeact word


In [5]:
failure_examples = [
    {"query": "I feel sick",    "misses": "I an unwell, patient has fever"},
    {"query": "How to cook rice",    "misses": "Steps to prepare rice"},
    {"query": "Vehicle speed",    "misses": "car acceleration, automobile velocity"},
    {"query": "ML model accuracy",    "misses": "performance of a machine learning model"},
]

for ex in failure_examples:
  print(f"Query: {ex['query']}")
  print(f"Answer: {ex['misses']}")
  print("-"*30)

Query: I feel sick
Answer: I an unwell, patient has fever
------------------------------
Query: How to cook rice
Answer: Steps to prepare rice
------------------------------
Query: Vehicle speed
Answer: car acceleration, automobile velocity
------------------------------
Query: ML model accuracy
Answer: performance of a machine learning model
------------------------------


In [6]:
#prerequisites for encoding for semantic search
#to get o.p - 384-dimensional vectors
#Note: first run downloads the model (~80MB).
print("Loading embedding model... (may take 1-2 minutes on first run)")

model = SentenceTransformer('all-MiniLM-L6-v2')

print("Model loaded successfully!")
print(f"Model produces vectors of size: {model.get_sentence_embedding_dimension()}")

Loading embedding model... (may take 1-2 minutes on first run)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded successfully!
Model produces vectors of size: 384


/tmp/ipykernel_3164/3894555355.py:9: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model produces vectors of size: {model.get_sentence_embedding_dimension()}")


In [7]:
#Encoding for semantic search
#Generate 1st text embedding

sentence = "ETL is used to clean and tansform data" #sentence to embed
embedding = model.encode(sentence)
#model.encode() converts string to a vector (list of numbers)

print(f"Input sentence: '{sentence}'")
print()
print(f"Embedding type: {type(embedding)}") #numpy.ndarray
print(f"Embedding shape: {embedding.shape}")#always 384 vectors
print()
print(f"First 10 numbers: {embedding[:10]}")#different decimal values for each vectors
print()
print(f"Min value: {embedding.min()}")
print(f"Max value: {embedding.max()}")

Input sentence: 'ETL is used to clean and tansform data'

Embedding type: <class 'numpy.ndarray'>
Embedding shape: (384,)

First 10 numbers: [-0.058379    0.03554146  0.01654611 -0.02895406  0.03935834 -0.0858812
  0.03790451 -0.00531707  0.0565182   0.02839596]

Min value: -0.13352325558662415
Max value: 0.17310813069343567


In [8]:
documents = [
    "ETL is used to clean and transform data",
    "A vehicle is a mode of transport",
    "Cars and truck are popular automobiles",
    "SQL is used to query databses",
    "Machine learning trains models on data",
    ]
query_keyword = "vehicle" #the word we are searching for

print("~="*30)
print(f"\t\tSEMANTIC SEARCH for '{query_keyword}'")
print("~="*30)
print()

# Encode documents and query
doc_embeddings = model.encode(documents)
query_embedding = model.encode(query_keyword)

# Calculate cosine similarities
# sentence_transformers.util.cos_sim returns a tensor of similarities
from sentence_transformers import util
similarities_tensor = util.cos_sim(query_embedding, doc_embeddings)[0]
print(f"Data type of cos_sim returns: {similarities_tensor.dtype}")

print(f"Query: '{query_keyword}'\n")
print("Semantic Search Results (ranked by similarity):")
threshold = 0.5
for i in range(len(similarities_tensor)):
  if(similarities_tensor[i] > threshold):
    flag = "FOUND"
  else:
    flag = "MISSED"
  print(f"  Score: {similarities_tensor[i]:.4f} - {flag} - [doc_{i}]: {documents[i]}")

print()
print("PROBLEM SOLVED! Semantic search correctly identifies 'Cars and truck' as relevant to 'vehicle'.")

~=~=~=~=~=~=~=~=~=~=~=~=~=~=~=~=~=~=~=~=~=~=~=~=~=~=~=~=~=~=
		SEMANTIC SEARCH for 'vehicle'
~=~=~=~=~=~=~=~=~=~=~=~=~=~=~=~=~=~=~=~=~=~=~=~=~=~=~=~=~=~=

Data type of cos_sim returns: torch.float32
Query: 'vehicle'

Semantic Search Results (ranked by similarity):
  Score: 0.0857 - MISSED - [doc_0]: ETL is used to clean and transform data
  Score: 0.6972 - FOUND - [doc_1]: A vehicle is a mode of transport
  Score: 0.5707 - FOUND - [doc_2]: Cars and truck are popular automobiles
  Score: 0.0553 - MISSED - [doc_3]: SQL is used to query databses
  Score: 0.2016 - MISSED - [doc_4]: Machine learning trains models on data

PROBLEM SOLVED! Semantic search correctly identifies 'Cars and truck' as relevant to 'vehicle'.


In [9]:
my_sentences = [
    "She is wearing a pretty black chudithar",
    "I planned today's outfit yesterday",
    "The newly released web series is becoming infamous for its misleading dialogues"
]

my_embeddings = model.encode(my_sentences)

def cosine_similarity(s1, s2):
  return util.cos_sim(s1, s2).item()
sim_01 = cosine_similarity(my_embeddings[0], my_embeddings[1])
sim_02 = cosine_similarity(my_embeddings[0], my_embeddings[2])

print("YOUR EXPERIMENT RESULTS")
print("-"*60)
print(f"Sentence A: '{my_sentences[0]}'")
print(f"Sentence B: '{my_sentences[1]}'")
print(f"Sentence C: '{my_sentences[2]}'")
print()
print(f"Similarity (A vs B):  {sim_01}")
print(f"Similarity (A vs C):  {sim_02}")
print()

YOUR EXPERIMENT RESULTS
------------------------------------------------------------
Sentence A: 'She is wearing a pretty black chudithar'
Sentence B: 'I planned today's outfit yesterday'
Sentence C: 'The newly released web series is becoming infamous for its misleading dialogues'

Similarity (A vs B):  0.25785189867019653
Similarity (A vs C):  0.04697011411190033



In [10]:
sentence =[
    "Machine Learning is used to predict the ouput based on trained data",
    "Types : Supervised and Unsupervised Learning",
    "I enjoy learning in the classroom",
]

embedding = model.encode(sentence)
query_keyword = "Machine Learning"
print(f"Query: {query_keyword}\n")

for i,doc in enumerate(sentence):
  print(f"Doc{i}: {doc}")
  embedding_query = model.encode(query_keyword.lower())
  embedding_doc = model.encode(doc.lower())
  similarity = util.cos_sim(embedding_query, embedding_doc).item()
  if similarity > 0.4:
    print(f" FOUND [doc_{i}]: {doc} (similarity={similarity:.2f})\n")
  else:
    print(f" MISSED [doc_{i}]: {doc} (similarity={similarity:.2f})\n")

Query: Machine Learning

Doc0: Machine Learning is used to predict the ouput based on trained data
 FOUND [doc_0]: Machine Learning is used to predict the ouput based on trained data (similarity=0.64)

Doc1: Types : Supervised and Unsupervised Learning
 FOUND [doc_1]: Types : Supervised and Unsupervised Learning (similarity=0.48)

Doc2: I enjoy learning in the classroom
 MISSED [doc_2]: I enjoy learning in the classroom (similarity=0.12)



chromaDB


In [13]:
chroma_client =chromadb.Client()
collection=chroma_client.get_or_create_collection("demo_notes")
#named container within a ChromaDB instance where you store a set of related embeddings.
#Each collection is isolated, meaning queries performed on one collection do not affect or search other collections.
# Each item you add to a collection typically comprises three primary elements:
  # Embedding (Vector): This is the numerical representation (a float vector, e.g., 384-dimensional) generated by an embedding model. It captures the semantic meaning of your data.
  # Document (Text): This is the original raw text or content from which the embedding was derived. While not strictly required for similarity search, it's crucial for understanding the context of retrieved embeddings.
  # Metadata (Dictionary): This is an optional dictionary of key-value pairs that provides additional descriptive attributes for the embedding. Metadata is invaluable for filtering and refining your searches (e.g., {'author': 'John Doe', 'category': 'fiction', 'year': 2023}).
  # ID (Unique Identifier): Each entry within a collection must have a unique identifier, often a string, which allows for specific retrieval, updating, or deletion of individual items.
#ChromaDB automatically builds an internal index (often using algorithms like Hierarchical Navigable Small World (HNSW)). This index is a complex data structure optimized for performing Approximate Nearest Neighbor (ANN) searches efficiently. Thus making it computationally effective

# Collections can reside in different storage backends:
    # In-Memory: Like chromadb.Client(), where the collection exists only for the duration of the process. This is good for quick experiments.
    # Persistent: Stored on disk (e.g., chromadb.PersistentClient()) so data is retained across sessions.
    # Client-Server (HTTP/Distributed): For larger, multi-user, or production deployments, collections typically reside on a dedicated ChromaDB server, accessed via an HTTP client.
    # In essence, a ChromaDB collection is a schema-flexible, indexed container for semantic data, enabling efficient storage and retrieval of information based on its contextual meaning rather than just keyword matching.

print("ChromaDB client created (in-memory mode)")

print(f"Collection name:demo_notes ")
print(f"Documents in collection :{collection.count()}")

ChromaDB client created (in-memory mode)
Collection name:demo_notes 
Documents in collection :0


In [14]:
sample_docs = [
    "ETL is data transformation pipeline",
    "A vehicle is a mode of transportation",
    "Cars and trucks are popular automoblies",
    "Extracts, Transform and Load",
    "Machine learning trains models o data"
]

sample_ids = ['doc001', 'doc002', 'doc003', 'doc004', 'doc005']

sample_metadata = [
    {"subject": "Data Engineering", "topic": "ETL"},
    {"subject": "Transportation", "topic": "Vehicles"},
    {"subject": "Transportation", "topic": "Automobiles"},
    {"subject": "Data Engineering", "topic": "Data Transformation"},
    {"subject": "Machine Learning", "topic": "Models"}
]

sample_embeddings = model.encode(sample_docs).tolist()

collection.add(
    documents=sample_docs,
    embeddings=sample_embeddings, # Pass the generated embeddings
    ids=sample_ids,
    metadatas=sample_metadata
)

print("Documents added successfully")
print(f"Number of documents in collection: {collection.count()}")

Documents added successfully
Number of documents in collection: 5


In [15]:
query="How do I clean and prepare data?"

results=collection.query(
    query_texts=[query],
    n_results=3
)

print("RESULT KEYS AVAILABLE:")
print(list(results.keys()))

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:01<00:00, 61.3MiB/s]


RESULT KEYS AVAILABLE:
['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas', 'distances']


In [16]:
print(f"Query :'{query}'")
print("="*60)
print()

matched_docs = results['documents'][0]
matched_ids = results['ids'][0]
matched_distance = results['distances'][0]
matched_metadata = results['metadatas'][0]

for rank,(doc , doc_id,dist,meta)in enumerate(zip(matched_docs,matched_ids,matched_distance,matched_metadata)):
  print(f"Rank:{rank}|ID:{doc_id}|Distance:{dist:.2f}")
  print(f"Subject:{meta['subject']}|Topic:{meta['topic']}")
  print(f"Document:{doc}\n")

Query :'How do I clean and prepare data?'

Rank:0|ID:doc001|Distance:1.62
Subject:Data Engineering|Topic:ETL
Document:ETL is data transformation pipeline

Rank:1|ID:doc005|Distance:1.70
Subject:Machine Learning|Topic:Models
Document:Machine learning trains models o data

Rank:2|ID:doc004|Distance:1.71
Subject:Data Engineering|Topic:Data Transformation
Document:Extracts, Transform and Load



In [17]:
filtered_results = collection.query(
    query_texts=[query],
    n_results=3,
    where={"topic": "ETL"}
)



print(f"FILTERED QUERY: {query}")
print("Filter only Data Engineering documents")
print("="*60)

for rank, (doc, dist, meta) in enumerate(zip(
    filtered_results['documents'][0],
    filtered_results['distances'][0],
    filtered_results['metadatas'][0]),start=1):
  print(f"Rank: {rank} | Distance: {dist:.2f} | Subject: {meta['subject']}")
  print(f"{doc}\n")

FILTERED QUERY: How do I clean and prepare data?
Filter only Data Engineering documents
Rank: 1 | Distance: 1.62 | Subject: Data Engineering
ETL is data transformation pipeline



In [18]:
print("DISTANCE TO SIMILARITY CONVERSION")
print(f"{'Distance':<15}{'Similarity':<15}{'Interpretation':<20}")
distances=[0.05,0.20,0.40,0.65,0.90]
interpretations=["Near identical","Very similar","Related","Somewhat related","Not related"]
for dist,interp in zip(distances,interpretations):
  similarity=1-dist
  print(f"{dist:<15.2f}{similarity:<15.2f}{interp:<20}")

DISTANCE TO SIMILARITY CONVERSION
Distance       Similarity     Interpretation      
0.05           0.95           Near identical      
0.20           0.80           Very similar        
0.40           0.60           Related             
0.65           0.35           Somewhat related    
0.90           0.10           Not related         
